# S3.5 — Complex JSON Handling
**Date completed:** September 2026  
**Status:** Completed  
**Key functions:** explode, posexplode, from_json, get_json_object

In [0]:
# ============================================================
# Cell 2 — explode(): One Row per Array Element
#
# Goal:
# 1. Understand how explode() converts array elements into rows.
# 2. Compare explode() with explode_outer().
# 3. Understand NULL arrays, empty arrays and NULL elements.
# 4. Demonstrate flattening RetailPulse order items.
# 5. Validate the output using record counts.
#
# Environment: Databricks Serverless
# Storage writes: None
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    ArrayType
)

print("=== S3.5: EXPLODE AND EXPLODE_OUTER ===")

# ------------------------------------------------------------
# Step 1 — Define the explicit schema
#
# items is an array containing string elements.
#
# nullable=True:
# The complete items column can be NULL.
#
# containsNull=True:
# Individual array elements can also be NULL.
# ------------------------------------------------------------

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer", StringType(), True),
    StructField("items", ArrayType(StringType(), containsNull=True))
])

# ------------------------------------------------------------
# Step 2 — Create RetailPulse sample data
#
# We are intentionally testing three edge cases:
#
# Order 1004: NULL array
# Order 1005: Empty array
# Order 1006: Array containing a NULL element
# ------------------------------------------------------------
data = [
    (1001, "Dhiraj", ["Mobile", "Charger", "Case"]),
    (1002, "Rahul",  ["Laptop"]),
    (1003, "Priya",  ["Shoes", "Bag"]),

    (1004, "Amit",   None),
    (1005, "Neha",   []),
    (1006, "Ravi",   [None])
]

df_orders_arrays = spark.createDataFrame(data, schema)

print("\n=== STEP 2: ORIGINAL DATA ===")
df_orders_arrays.printSchema()
df_orders_arrays.show(truncate = False)

source_count = df_orders_arrays.count()
print(f"Original row count: {source_count}")

# ------------------------------------------------------------
# Step 3 — explode()
#
# Converts each array element into a separate output row.
#
# NULL array  → No output row
# Empty array → No output row
# [None]      → One output row containing NULL
# ------------------------------------------------------------

print("\n=== STEP 3: EXPLODE ===")

df_exploded = df_orders_arrays.select(
    F.col("order_id"),
    F.col("customer"),
    F.explode(F.col("items")).alias("items")
)

df_exploded.show(truncate = False)
explode_count = df_exploded.count()
print(f"Row count after explode(): {explode_count}")

# ------------------------------------------------------------
# Step 4 — explode_outer()
#
# Preserves parent records with NULL or empty arrays.
#
# NULL array  → One output row containing NULL
# Empty array → One output row containing NULL
# [None]      → One output row containing NULL
# ------------------------------------------------------------

print("\n=== STEP 4: EXPLODE_OUTER ===")

df_exploded_outer = df_orders_arrays.select(
    F.col("order_id"),
    F.col("customer"),
    F.explode_outer(F.col("items")).alias("items")
)

df_exploded_outer.show(truncate = False)
explode_outer_count = df_exploded_outer.count()

print(f"Row count after explode_outer():"
      f"{explode_outer_count}"
)

# ------------------------------------------------------------
# Step 5 — Compare record counts
#
# Expected:
#
# Source Data     = 6 rows
# explode()       = 7 rows
# explode_outer() = 9 rows
#
# These counts are specific to our demonstration dataset.
# ------------------------------------------------------------

print("\n=== STEP 5: ROW COUNT VALIDATION ===")

print(f"Original records       : {source_count}")
print(f"After explode()        : {explode_count}")
print(f"After explode_outer()  : {explode_outer_count}")

assert source_count == 6, "Unexpected source record count"
assert explode_count == 7, "Unexpected explode result"
assert explode_outer_count == 9, "Unexpected explode_outer result"

print("\nAll record-count validations passed.")

# ------------------------------------------------------------
# Step 6 — Final Learning Summary
# ------------------------------------------------------------
print("\n=== FINAL LEARNING SUMMARY ===")

print("""
explode():
- Creates one row per array element.
- Produces no output row for a NULL array.
- Produces no output row for an empty array.
- Preserves NULL elements within an existing array.

explode_outer():
- Creates one row per array element.
- Retains NULL arrays as one row containing NULL.
- Retains empty arrays as one row containing NULL.
- Preserves NULL elements within an existing array.

Enterprise Consideration:
- Select the function based on the downstream business requirement.
- Use explode_outer() when parent-record retention is required.
- NULL handling and data-quality validation are separate decisions.
""")

print("=== CELL 2 COMPLETED ===")

# posexplode() — position + element:

In [0]:
# ============================================================
# Cell 4 — posexplode(): Position Index + Element
#
# Goal:
# 1. Understand how posexplode() works.
# 2. Compare explode() with posexplode().
# 3. Identify the first item from each order.
# 4. Validate output record counts.
#
# Environment: Databricks Serverless
# Storage writes: None
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    ArrayType
)

print("=== S3.5: POSEXPLODE ===")


# ------------------------------------------------------------
# Step 1 — Create sample RetailPulse data
# ------------------------------------------------------------

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer", StringType(), True),
    StructField(
        "items",
        ArrayType(StringType(), containsNull=True),
        True
    )
])

data = [
    (1001, "Dhiraj", ["Mobile", "Charger", "Case"]),
    (1002, "Rahul", ["Laptop"]),
    (1003, "Priya", ["Shoes", "Bag"])
]

df_orders_pos = spark.createDataFrame(data, schema)

print("\n=== STEP 1: ORIGINAL DATA ===")

df_orders_pos.show(truncate=False)

source_count = df_orders_pos.count()

print(f"Source row count: {source_count}")


# ------------------------------------------------------------
# Step 2 — explode()
#
# Returns one output column containing array elements.
# ------------------------------------------------------------

print("\n=== STEP 2: EXPLODE ===")

df_explode_demo = df_orders_pos.select(
    "order_id",
    "customer",
    F.explode("items").alias("item")
)

df_explode_demo.show(truncate=False)


# ------------------------------------------------------------
# Step 3 — posexplode()
#
# Returns two output columns:
#
# position = Original array index (starts from zero)
# item     = Array element
# ------------------------------------------------------------

print("\n=== STEP 3: POSEXPLODE ===")

df_pos = df_orders_pos.select(
    F.col("order_id"),
    F.col("customer"),
    F.posexplode("items").alias("position", "item")
)

df_pos.show(truncate=False)

position_count = df_pos.count()

print(f"Row count after posexplode(): {position_count}")


# ------------------------------------------------------------
# Step 4 — Real use case: Identify the first item
#
# position == 0 represents the first array element.
# ------------------------------------------------------------

print("\n=== STEP 4: FIRST ITEM PER ORDER ===")

df_first_item = (
    df_pos
    .filter(F.col("position") == 0)
    .select(
        "order_id",
        "customer",
        F.col("item").alias("first_item")
    )
)

df_first_item.show(truncate=False)


# ------------------------------------------------------------
# Step 5 — Validate results
#
# 3 source orders.
# 6 total array elements.
# 3 first-item records.
#
# Expected counts apply to this demonstration dataset only.
# ------------------------------------------------------------

print("\n=== STEP 5: VALIDATION ===")

assert source_count == 3, "Unexpected source count"

assert position_count == 6, "Unexpected posexplode count"

assert df_first_item.count() == 3, "Unexpected first-item count"

print("All validations passed.")


# ------------------------------------------------------------
# Step 6 — Final Learning Summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("""
explode():
- Expands array elements into individual rows.
- Returns the element, without its position.

posexplode():
- Expands array elements into individual rows.
- Returns both the position and the element.
- Array positions start from zero.

Practical use:
- Identify the first item from an array.
- Preserve the original sequence information.
- Analyse ordered lists from JSON and API responses.

Important:
- posexplode() does not guarantee DataFrame row display order.
- Use orderBy() when an ordered result is required.
- posexplode() produces no row for NULL or empty arrays.
""")

print("=== CELL 3 COMPLETED ===")

In [0]:
# ============================================================
# Cell 5 — from_json() and get_json_object()
#
# Goal:
# 1. Parse JSON stored inside a STRING column.
# 2. Extract individual fields using get_json_object().
# 3. Parse JSON into a STRUCT using from_json().
# 4. Flatten structured JSON into individual columns.
# 5. Compare output data types and handle JSON null values.
#
# Environment: Databricks Serverless
# Storage writes: None
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

print("=== S3.5: FROM_JSON AND GET_JSON_OBJECT ===")

# ------------------------------------------------------------
# Step 1 — Create sample data
#
# The payload column contains JSON as text (StringType).
#
# This is representative of decoded JSON messages received
# from an API or a message-streaming source.
# ------------------------------------------------------------

raw_data = [
    (1, '{"customer_id": 202, "city": "Delhi","order_value": 2500.00}'),
    (2, '{"customer_id": 202, "city": "Mumbai", "order_value": 1800.0}'),
    (3, '{"customer_id": 203, "city": "Pune", "order_value": 3200.0}'),
    (4, '{"customer_id": 204, "city": null, "order_value": null}')
]

df_raw = spark.createDataFrame(
    raw_data,["event_id", "payload"]
)

print("\n=== STEP 1: RAW JSON STRING DATA ===")

df_raw.printSchema()
df_raw.show(truncate = False)

# ------------------------------------------------------------
# Step 2 — get_json_object()
#
# Extracts a selected value using a JSON path.
#
# IMPORTANT:
# Extracted values are returned as StringType.
# JSON null values return SQL NULL.
# ------------------------------------------------------------

print("\n=== STEP 2: GET_JSON_OBJECT ===")

df_single = df_raw.select(

    F.col("event_id"),

    F.get_json_object(
        F.col("payload"),
        "$.customer_id"
    ).alias("customer_id"),

    F.get_json_object(
        F.col("payload"),
        "$.city"
    ).alias("city"),

    F.get_json_object(
        F.col("payload"),
        "$.order_value"
    ).alias("order_value")

)

df_single.show(truncate = False)
print("\nSchema after get_json_object():")
df_single.printSchema()

# ------------------------------------------------------------
# Step 3 — Define the expected JSON schema
#
# from_json() requires a schema describing the JSON structure.
#
# This allows Spark to produce properly typed fields.
# ------------------------------------------------------------

json_schema = StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("city", StringType(), True),
    StructField("order_value", DoubleType(), True)])

# ------------------------------------------------------------
# Step 4 — Parse JSON using from_json()
#
# Converts JSON text into a structured column.
#
# The new data column will have StructType.
# ------------------------------------------------------------

print("\n=== STEP 4: FROM_JSON ===")

df_parsed = df_raw.select(
    F.col("event_id"),
    F.from_json(F.col("payload"),json_schema).alias("data"))

print("\nParsed JSON schema:")
df_parsed.printSchema()
print("\nParsed JSON data:")
df_parsed.show(truncate=False)

# ------------------------------------------------------------
# Step 5 — Flatten the parsed STRUCT
#
# Nested fields can be accessed using dot notation.
#
# data.customer_id
# data.city
# data.order_value
# ------------------------------------------------------------

print("\n=== STEP 5: FLATTEN STRUCT ===")

df_flat = df_parsed.select(
    F.col("event_id"),
    F.col("data.customer_id").alias("customer_id"),
    F.col("data.city").alias("city"),
    F.col("data.order_value").alias("order_value")
)

df_flat.show(truncate=False)

print("\nFlattened schema:")

df_flat.printSchema()

# ------------------------------------------------------------
# Step 6 — Validate record counts and data types
#
# Expected source rows = 4
#
# These assertions apply to the learning dataset only.
# ------------------------------------------------------------

print("\n=== STEP 6: VALIDATION ===")

source_count = df_raw.count()
single_count = df_single.count()
parsed_count = df_parsed.count()
flat_count = df_flat.count()

print(f"Source records            : {source_count}")
print(f"get_json_object() records : {single_count}")
print(f"from_json() records       : {parsed_count}")
print(f"Flattened records         : {flat_count}")


assert source_count == 4, "Unexpected source count"
assert single_count == source_count, ("get_json_object() record count mismatch")
assert parsed_count == source_count, ("from_json() record count mismatch")
assert flat_count == source_count, ("Flattened record count mismatch")

# Verify that get_json_object() returns strings.
assert isinstance(
    df_single.schema["customer_id"].dataType,
    StringType
), "get_json_object() customer_id should be StringType"

# Verify that from_json() follows the explicit schema.
assert isinstance(
    df_flat.schema["customer_id"].dataType,
    IntegerType
), "Parsed customer_id should be IntegerType"

assert isinstance(
    df_flat.schema["order_value"].dataType,
    DoubleType
), "Parsed order_value should be DoubleType"

print("\nAll record-count and data-type validations passed.")

# ------------------------------------------------------------
# Step 7 — Inspect JSON NULL handling
#
# Event 4 contains explicit JSON null values.
# These become SQL NULL after parsing.
# ------------------------------------------------------------

print("\n=== STEP 7: JSON NULL HANDLING ===")

df_flat.filter(
    F.col("event_id") == 4
).show(truncate = False)

# ------------------------------------------------------------
# Step 8 — Final Learning Summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("""
get_json_object():
- Extracts selected values from JSON text.
- Uses JSON path expressions such as $.customer_id.
- Returns extracted values as strings.
- Useful when only specific JSON fields are required.

from_json():
- Parses JSON text using a supplied schema.
- Produces a structured column.
- Preserves data types defined in the schema.
- Useful for nested and structured JSON processing.

Important:
- JSON null values become SQL NULL.
- Missing or malformed fields require appropriate validation.
- Parsing JSON does not automatically guarantee data quality.
- Performance depends on the workload and execution plan.
""")

print("=== CELL 4 COMPLETED ===")

In [0]:
# ============================================================
# Cell 5 — Real Enterprise Use Case: Nested JSON Flattening
#
# Goal:
# 1. Read real nested JSON from the Databricks sample dataset.
# 2. Inspect the source schema before transformation.
# 3. Flatten ordered_products into individual product rows.
# 4. Extract fields from nested product structures.
# 5. Validate the resulting product-level record count.
#
# Learning pattern:
# Raw Nested JSON → Product-Level Flat DataFrame
#
# Environment: Databricks Serverless
# Storage writes: None
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    ArrayType,
    StructType
)

print("=== S3.5: REAL NESTED JSON FLATTENING ===")


# ------------------------------------------------------------
# Step 1 — Read sample sales_orders JSON
#
# JSON schema inference is enabled by default.
# We do not need the CSV-specific inferSchema option.
# ------------------------------------------------------------

source_path = "dbfs:/databricks-datasets/retail-org/sales_orders/"

df_sales = (
    spark.read
    .format("json")
    .load(source_path)
)

print("\n=== STEP 1: RAW SALES ORDERS ===")

source_count = df_sales.count()

print(f"Raw sales orders: {source_count:,}")

df_sales.printSchema()

df_sales.show(3, truncate=50)


# ------------------------------------------------------------
# Step 2 — Inspect ordered_products
#
# Confirm the nested structure before accessing its fields.
# ------------------------------------------------------------

print("\n=== STEP 2: INSPECT ORDERED_PRODUCTS ===")

required_top_fields = {
    "order_number",
    "customer_id",
    "customer_name",
    "order_datetime",
    "ordered_products"
}

missing_top_fields = (
    required_top_fields - set(df_sales.columns)
)

if missing_top_fields:
    raise ValueError(
        f"Missing source fields: {sorted(missing_top_fields)}"
    )

df_sales.select(
    "order_number",
    "ordered_products"
).show(3, truncate=False)


# ------------------------------------------------------------
# Step 3 — Verify the nested product schema
#
# Expected:
# ordered_products = ArrayType(StructType(...))
#
# This prevents us from referencing fields that do not exist.
# ------------------------------------------------------------

print("\n=== STEP 3: VERIFY NESTED PRODUCT SCHEMA ===")

items_type = df_sales.schema["ordered_products"].dataType

if not isinstance(items_type, ArrayType):
    raise TypeError(
        "ordered_products is not an ArrayType. "
        "Inspect the source schema."
    )

product_type = items_type.elementType

if not isinstance(product_type, StructType):
    raise TypeError(
        "ordered_products does not contain StructType elements."
    )

print("Available product fields:")
print(product_type.fieldNames())

required_product_fields = {
    "id",
    "name",
    "price",
    "qty",
    "curr",
    "promotion_info"
}

missing_product_fields = (
    required_product_fields - set(product_type.fieldNames())
)

if missing_product_fields:
    raise ValueError(
        f"Missing product fields: {sorted(missing_product_fields)}"
    )

promotion_type = product_type["promotion_info"].dataType

if not isinstance(promotion_type, StructType):
    raise TypeError(
        "promotion_info is not a StructType. "
        "Inspect its actual structure."
    )

if "promo_disc" not in promotion_type.fieldNames():
    raise ValueError(
        "promo_disc does not exist inside promotion_info."
    )

print("Nested schema validation passed.")


# ------------------------------------------------------------
# Step 4 — Calculate expected product count
#
# Each array element should produce one product row.
#
# NULL and empty arrays produce zero rows with explode().
# ------------------------------------------------------------

print("\n=== STEP 4: EXPECTED PRODUCT COUNT ===")

expected_product_count = (
    df_sales
    .select(
        F.when(
            F.col("ordered_products").isNull(),
            F.lit(0)
        )
        .otherwise(F.size("ordered_products"))
        .alias("product_count")
    )
    .agg(
        F.sum("product_count").alias("expected_products")
    )
    .first()["expected_products"]
)

expected_product_count = expected_product_count or 0

print(f"Expected product rows: {expected_product_count:,}")


# ------------------------------------------------------------
# Step 5 — Explode ordered_products
#
# One order containing three products produces three rows.
#
# explode() does not retain orders with NULL or empty arrays.
# ------------------------------------------------------------

print("\n=== STEP 5: EXPLODE ORDERED_PRODUCTS ===")

df_exploded = df_sales.select(

    F.col("order_number"),
    F.col("customer_id"),
    F.col("customer_name"),
    F.col("order_datetime"),

    F.explode(
        F.col("ordered_products")
    ).alias("product")

)

exploded_count = df_exploded.count()

print(f"Rows after explode(): {exploded_count:,}")

df_exploded.printSchema()

df_exploded.show(3, truncate=50)


# ------------------------------------------------------------
# Step 6 — Flatten the product struct
#
# Access fields using dot notation.
#
# product.id
# product.name
# product.promotion_info.promo_disc
# ------------------------------------------------------------

print("\n=== STEP 6: FLATTEN PRODUCT FIELDS ===")

df_flat = df_exploded.select(

    F.col("order_number"),

    F.col("customer_id"),

    F.col("customer_name"),

    F.col("order_datetime"),

    F.col("product.id").alias("product_id"),

    F.col("product.name").alias("product_name"),

    F.col("product.price").alias("price"),

    F.col("product.qty").alias("quantity"),

    F.col("product.curr").alias("currency"),

    F.col(
        "product.promotion_info.promo_disc"
    ).alias("promo_discount")

)

print("\nFlattened product schema:")

df_flat.printSchema()

print("\nSample flattened records:")

df_flat.show(5, truncate=40)


# ------------------------------------------------------------
# Step 7 — Validate transformation
#
# Expected product elements should match exploded rows.
#
# Flattening struct fields should not change the row count.
# ------------------------------------------------------------

print("\n=== STEP 7: RECORD COUNT VALIDATION ===")

flat_count = df_flat.count()

print(f"Source order rows       : {source_count:,}")
print(f"Expected product rows   : {expected_product_count:,}")
print(f"After explode()         : {exploded_count:,}")
print(f"After struct flattening : {flat_count:,}")

assert exploded_count == expected_product_count, ("Exploded record count does not match expected product count")

assert flat_count == exploded_count, ("Flattening unexpectedly changed the record count")

print("\nAll record-count validations passed.")

# ------------------------------------------------------------
# Step 8 — Final Learning Summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("""
1. Read raw nested JSON using spark.read.json().
2. Inspect the actual source schema before transformation.
3. Use explode() to convert each array element into a row.
4. Use dot notation to extract fields from a nested struct.
5. Validate product-level record counts.
6. The resulting DataFrame demonstrates a Bronze-to-Silver
   flattening pattern.
7. No Silver table has been created or persisted in this cell.
""")

print("=== CELL 5 COMPLETED ===")

## Cell 5 — Real Nested JSON Flattening: Key Takeaways

- Read real nested JSON using `spark.read.json()`.
- Use `printSchema()` to inspect the source structure.
- Use `isinstance()` to verify expected data types before transformation.
- Use `explode()` to convert array elements into individual rows.
- Use dot notation to extract nested StructType fields.
- Use `count()` and `assert` to validate transformation results.
- Understand that one order containing multiple products produces multiple output rows.
- Demonstrated the Bronze → Silver flattening pattern.

**Important:** The transformation creates a flattened DataFrame; it does not physically save a Silver table.

**Enterprise learning:** Always inspect the source schema and validate the resulting data before loading it into Silver.

# S3.5 — Complex JSON Handling: Final Key Takeaways

## 1. Why Do We Learn Complex JSON Handling?

Real-world API responses, application events and source files often contain:

- Nested JSON objects
- Arrays containing multiple elements
- Arrays of structured records
- Optional and missing fields
- JSON stored as a STRING column

Our objective is to understand these structures and transform them into usable DataFrames.

---

## 2. Core Functions

| Function | Purpose | When to Use |
|---|---|---|
| explode() | Expands array or map elements into separate rows | When individual elements are required as rows |
| explode_outer() | Expands elements while retaining NULL and empty collections | When retaining the parent record is required |
| posexplode() | Returns array position and element | When the original element position matters |
| get_json_object() | Extracts values from JSON text using a JSON path | When selected JSON fields are needed |
| from_json() | Parses JSON text according to a supplied schema | When a structured and typed result is required |

---

## 3. explode() vs explode_outer()

| Input Array | explode() | explode_outer() |
|---|---|---|
| ["Mobile", "Case"] | 2 rows | 2 rows |
| NULL | 0 rows | 1 NULL row |
| [] | 0 rows | 1 NULL row |
| [NULL] | 1 NULL row | 1 NULL row |

### Important Learning

- explode() does not produce rows for NULL or empty arrays.
- explode_outer() retains NULL and empty arrays as one NULL output row.
- Both preserve NULL elements inside an existing array.
- Choose the function according to the business requirement.

NULL handling and data-quality validation are separate responsibilities.

---

## 4. posexplode() — Position and Element

Example input:

```python
["Mobile", "Charger", "Case"]
```

Expected result:

| position | item |
|---|---|
| 0 | Mobile |
| 1 | Charger |
| 2 | Case |

Important:

- Array positions start from zero.
- posexplode() preserves the original array position.
- It does not guarantee the display order of DataFrame rows.
- Use orderBy() when an ordered result is needed.

---

## 5. from_json() vs get_json_object()

### get_json_object()

```python
F.get_json_object(
    F.col("payload"),
    "$.customer_id"
)
```

- Extracts a specific value from JSON text.
- Returns the extracted value as StringType.
- Does not automatically provide the original JSON field's numeric data type.

### from_json()

```python
F.from_json(
    F.col("payload"),
    json_schema
)
```

- Parses JSON text using a defined schema.
- Produces a structured result.
- Applies the data types specified in the supplied schema.
- Supports complex nested structures.

### Enterprise Learning

Use from_json() with an explicit schema when a governed,
structured and typed representation of JSON data is required.

Use get_json_object() when selected values need to be extracted
directly from JSON text.

Neither function automatically guarantees data quality.

---

## 6. Struct Access and Nested Fields

After applying explode() to an array of structs, access nested
fields using dot notation.

Examples:

```python
F.col("product.id").alias("product_id")

F.col("product.name").alias("product_name")

F.col("product.price").alias("price")

F.col("product.qty").alias("quantity")

F.col(
    "product.promotion_info.promo_disc"
).alias("promo_discount")
```

Nested field names must be verified against the actual source schema.

---

## 7. Real Practical — sales_orders JSON

Source:

Databricks retail-org sales_orders sample JSON.

### Actual Execution Results

| Metric | Result |
|---|---:|
| Original sales orders | 4,074 |
| Expected product elements | 8,137 |
| After explode(ordered_products) | 8,137 |
| After flattening product struct | 8,137 |
| Record-count validation | PASSED |

### Transformation Process

Raw nested JSON
        ↓
Inspect source schema
        ↓
Verify ArrayType and StructType
        ↓
Calculate expected product count
        ↓
explode(ordered_products)
        ↓
Extract nested fields using dot notation
        ↓
Flattened product-level DataFrame
        ↓
Validate actual vs expected counts

### Important Understanding

The source contains 4,074 order records.

After exploding ordered_products, it contains 8,137 product records.

This increase is expected because an order can contain multiple products.

The target grain is:

**One row per product entry within an order.**

---

## 8. Validation Best Practices

### Count Validation

1. Calculate the expected number of product elements from source arrays.
2. Apply explode().
3. Compare the exploded count against the expected product count.
4. Flatten nested fields.
5. Verify that selecting nested fields has not unexpectedly changed the row count.

Example:

```python
assert exploded_count == expected_product_count, \
    "Product count mismatch"

assert flat_count == exploded_count, \
    "Flattening changed the record count"
```

### Understanding assert and isinstance

| Python Function | Purpose |
|---|---|
| isinstance() | Checks whether an object has the expected type |
| assert | Checks whether an expected condition is true |

Example:

```python
assert isinstance(items_type, ArrayType), \
    "Expected an ArrayType"
```

### Important Production Consideration

Matching record counts is necessary for this transformation's
reconciliation, but it does not prove complete data quality.

Additional checks may include:

- Missing product IDs
- Duplicate order-line records
- Invalid quantities
- Invalid prices
- Unexpected NULL values
- Incorrect data types
- Schema changes

---

## 9. Bronze-to-Silver Transformation Pattern

### Bronze

Preserve the raw nested JSON and available source metadata.

### Transformation

- Inspect schema.
- Parse JSON where required.
- Expand arrays.
- Extract nested fields.
- Apply data-type conversions.
- Validate expected records and business rules.

### Silver

Produce a structured and validated product-level dataset.

Our current Cell 5 created a flattened DataFrame in memory.

It has NOT physically created or written a Silver table.

Confirmed Databricks namespace:

workspace.default

Potential future learning table:

workspace.default.s35_order_items_demo

Table creation is pending storage/permission verification.

---

## 10. Important Production Considerations

- The source order_datetime is currently stored as StringType.
- Convert Unix timestamp values appropriately during Silver preparation.
- Validate quantities and prices before downstream consumption.
- Confirm business rules before replacing NULL promotion values.
- Keep sufficient identifiers to preserve the intended order-line grain.
- Validate the source-to-target transformation before persisting data.

---

## Final Interview Takeaways

**Q1. What does explode() do?**

Converts each element of an array or map into a separate row.

**Q2. What is the difference between explode() and explode_outer()?**

explode_outer() retains NULL and empty collections as a row,
while explode() produces no rows for them.

**Q3. What does posexplode() provide additionally?**

The original zero-based position of each array element.

**Q4. Why use from_json()?**

To parse JSON text into a structured result using a defined schema.

**Q5. Why use isinstance()?**

To verify that an object has the expected type before processing it.

**Q6. Why validate expected product counts?**

To detect unexpected record-count changes during array expansion
and nested-field extraction.

---

## Final Learning Principle

Understand the nested structure first.

Then parse, explode and flatten according to the required target grain.

Validate the transformation before writing the resulting dataset.

**S3.5 Practical Status: Completed and validated.**

**Physical Silver-table persistence: Not yet performed.**